[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/22_weight_init_solution.ipynb)

# 🟢 Solution: Kaiming Initialization

*Core Ops & Layers · Easy*

Reference implementation. Try it yourself in `22_weight_init.ipynb` first.

---
Implement the two classic weight initialisers as one plain function, selected by
a string `mode`.

$$\sigma_{\text{xavier}} = \sqrt{\frac{2}{f_{in} + f_{out}}}
\qquad
\sigma_{\text{he}} = \sqrt{\frac{2}{f_{in}}}$$

Four modes: `"xavier_normal"`, `"xavier_uniform"`, `"he_normal"`, `"he_uniform"`.
(Glorot is another name for Xavier; Kaiming is another name for He.)

### Rules
- Signature: `init_weights(key, shape, mode="he_normal") -> jnp.ndarray`
- `shape` is `(fan_in, fan_out)` — the JAX layout, since the forward pass is `x @ W`
- Do **not** use `jax.nn.initializers` or `flax.nnx.initializers`
- Normal variants: zero-mean Gaussian with standard deviation $\sigma$
- Uniform variants: $U(-a, a)$ with $a$ chosen so the **variance matches** the
  normal variant
- Unknown `mode` must raise `ValueError`
- Pure and deterministic: the same key must give the same array

### The derivation, in three lines
Take one layer $y = xW$ with $f_{in}$ inputs, all entries i.i.d. and zero-mean:

$$\mathrm{Var}(y) = f_{in}\,\mathrm{Var}(W)\,\mathrm{Var}(x)$$

so the forward pass keeps its scale when $\mathrm{Var}(W) = 1/f_{in}$. Running the
same argument through the backward pass — where the gradient flows through $W^\top$
and therefore sees $f_{out}$ terms — demands $\mathrm{Var}(W) = 1/f_{out}$. You
cannot have both unless the layer is square, so **Xavier splits the difference**
with the harmonic-style compromise $2/(f_{in} + f_{out})$.

He then adds one observation: ReLU zeroes half its inputs, so it destroys half the
variance at every layer. Doubling the numerator, $2/f_{in}$, exactly cancels that.

### Which pairs with which
| Init | Activation | Why |
|---|---|---|
| Xavier | tanh, sigmoid, linear/attention projections | roughly linear and symmetric near 0, no variance lost |
| He | ReLU, GELU, SiLU | compensates for the half of the signal the gate kills |

Use Xavier with a 30-layer ReLU net and every layer shrinks the signal by
$1/\sqrt{2}$; after 20 layers the activations are down by ~$2^{-10}$ and the
gradient with them. That is the failure the tests below reproduce, and it is the
reason "we couldn't train deep nets before 2015" is only half a story about
architectures.

### Uniform vs normal
$\mathrm{Var}(U(-a,a)) = a^2/3$, so matching a target $\sigma$ needs
$a = \sqrt{3}\,\sigma$ — i.e. $a = \sqrt{6/(f_{in}+f_{out})}$ for Xavier. That
$\sqrt{6}$ in every framework's `xavier_uniform` comes from precisely here, not
from anywhere magic.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import math

import jax
import jax.numpy as jnp


def init_weights(key, shape, mode="he_normal"):
    fan_in, fan_out = shape          # x @ W, so fan_in is the leading axis

    if mode in ("xavier_normal", "xavier_uniform"):
        std = math.sqrt(2.0 / (fan_in + fan_out))
    elif mode in ("he_normal", "he_uniform"):
        std = math.sqrt(2.0 / fan_in)
    else:
        raise ValueError(
            f"unknown mode {mode!r}; expected one of "
            "'xavier_normal', 'xavier_uniform', 'he_normal', 'he_uniform'"
        )

    if mode.endswith("_normal"):
        return jax.random.normal(key, shape) * std

    # Var(U(-a, a)) = a^2 / 3, so a = sqrt(3) * std matches the normal variant.
    limit = math.sqrt(3.0) * std
    return jax.random.uniform(key, shape, minval=-limit, maxval=limit)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

key = jax.random.key(0)
for mode in ("xavier_normal", "xavier_uniform", "he_normal", "he_uniform"):
    w = init_weights(key, (256, 64), mode)
    print(f"{mode:>15s}  std={float(jnp.std(w)):.4f}  max={float(jnp.abs(w).max()):.4f}")

print("\ntarget xavier std:", (2 / (256 + 64)) ** 0.5)
print("target he     std:", (2 / 256) ** 0.5)

# Signal propagation through 20 ReLU layers, He vs Xavier.
x = jax.random.normal(jax.random.key(1), (64, 256))
for mode in ("he_normal", "xavier_normal"):
    h, k = x, jax.random.key(2)
    for _ in range(20):
        k, sub = jax.random.split(k)
        h = jax.nn.relu(h @ init_weights(sub, (256, 256), mode))
    print(f"{mode:>15s}  rms after 20 layers: {float(jnp.sqrt(jnp.mean(h ** 2))):.6f}")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("weight_init")